# Importeren van libraries

In [ ]:
import pandas as pd
#Pandas om dataframes te maken en analyseren
import pyarrow
#Pyarrow module om .csv's na cleaning en validatie in parquet-formaat op te slaan.
from pathlib import Path
#Path om binnen de omgeving .csv-files te zoeken en laden.
import re
#Regex module importeren om pattern matching te gebruiken.
import numpy as np
#Numpy module voor matrix berekeningen. Voornamelijk gebruikt om vergelijkingen te maken met np.isclose.
from shared import init_analyse
#Uit shared py de init_analyse functie. 
#Een kleine functie op basis van de eerste analyse waarbij dezelfde functies binnen pandas op 1 dataframe worden gebruikt.
pd.options.display.float_format = "{:,.2f}".format
#Floats weergeven als getallen met 2 decimalen achter de komma, aanvankelijk neigt pandas bij grotere getallen naar wetenschappelijke weergaven.

# 1 ***Analyse sales data***

In dit onderdeel importeren en analyseren we de salesdata met behulp van pandas

## 1.1 Importeren en inlezen dataset

In [ ]:
sales_path= Path(r"Data\Contoso_Sales.csv")
#Inlezen sales.csv
sales_df = pd.read_csv(sales_path,
                       decimal=",")
#Dataframe koppelen aan variabele
#Decimal argument meegegeven, omdat pandas standaard "." als delimiter ziet.

Data is met behulp van pathlib en pandas ingelezen.  
Bij het inlezen ontstond initieel een dtypewarning voor de kolommen ReturnAmount en DiscountAmount. 
Door bij het inlezen het decimal teken mee te geven wordt deze fout verholpen en worden floats gelijk correct ingelezen.

## 1.2. Eerste analyse
Hier wordt de algehele dataset geanalyseerd op afwijkingen, waar hierna op groter detailniveau naar gekeken wordt.

In [ ]:
sales_df.info()
#Eerste data over kolommen opvragen

- Datekey wordt als string weergegeven, deze dient nog omgezet te worden naar een tijdreeks.
- UnitCost, UnitPrice, TotalCost en SalesAmount werden initieel als strings geïnterpreteerd. Dit is afgevangen door bij de import de parameter decimal mee te geven en komma's aan te merken als het decimaal-teken.

In [ ]:
sales_df.head(20)
#De eerste 20 rijn tonen met waarden.

Datekey lijkt een datumwaarde te moeten zijn. Op het moment is dit een string. Deze dient bij het opschonen omgezet te worden, zodat deze als key gebruikt kan worden.  
UnitCost, UnitPrice, TotalCost en SalesAmount lijkt om bedragen te gaan. De stuksprijs maal de quantity is hier ook gelijk aan de Totalcost en SalesAmount.  
Gezien het feit er geen strings in de tabel zitten, is er geen noodzaak om te controleren of deze afwijken van elkaar in schrijfwijze. Hiermee wordt bedoeld waarden die dezelfde categorie betreffen, maar andere schrijfwijze hebben door gebruik van hoofdletters en leestekens.   
ChannelKey is niet in constante camelcase geschreven met de rest van de set.

In [ ]:
sales_df.tail(20)
#Laatste 20 waarden bekijken.

Laatste 20 rijen laten hetzelfde patroon zien. In deze rijen zitten ook returns verwerkt. Deze lijken wel goed verwerkt te worden in de TotalCost, bijvoorbeeld 10 items verkocht - 1 return = 9 * UnitCost.  
Op salesniveau lijkt dit echter niet correct doorberekend. De SalesAmount blijft gelijk aan 10 maal de UnitPrice, ondanks de return.  
Huidige actie lijkt  om een extra kolom toe te voegen die de SalesAmount berekend met inachtname van Returns. Dit houdt de originele kolom in tact, maar geeft een betere representatie van de daadwerkelijke omzet van een order.

In [ ]:
sales_df.describe()

In [ ]:
sales_df.isna().sum()
#Controleren op lege waarden in cellen. Dit is niet van toepassing.


In [ ]:
sales_df.nunique() 
#Aantal unique waarden opsommen.

In [ ]:
sales_df[sales_df.duplicated()]
#Checken op duplicated rows

Op basis van bovenstaande analyses zijn er geen lege waardes of duplicaten aanwezig in deze dataset.

De volgende keys dienen gecontroleerd te worden op aanwezigheid in de andere dataset:
- 4 verkoopkanalen volgens sales set.
- 306 unieke winkels volgens sales set.
- 1689 soort producten
- 28 promoties
- 783 retourregels, hoewel er geen verdere datasets met betrekking tot retourproducten lijkt te zijn.

### **Tussentijdse bevindingen**

De sales dataset loopt van 1 januari 2011 tot en met 31 december 2013 en bestaat uit ***2.282.482*** unieke saleskeys.
Het is nog onduidelijk of de return amounts verrekend moeten worden met de SalesAmount, gezien het verschil tussen kosten en opbrengsten bij een retour.  
Het overzicht op maandniveau laat zien dat de brutomarge redelijk constant is over de gegeven periode. Het aantal verkopen lijkt wel af te nemen.
De datetimes zijn reeds omgezet.

De volgende keys dienen gecontroleerd te worden op aanwezigheid in de andere dataset:
- 4 **verkoopkanalen** volgens sales set.
- 306 **unieke winkels** volgens sales set.
- 1689 soort **producten**
- 28 **promoties**
- 783 **retourregels**

Een eerste run van de validatie leverde een error op bij het controleren van het verband tussen UnitCost, SalesQuantity en ReturnQuantity.   
Uit de tail data uit de initiële analyse leek het logisch om aan te nemen dat alle TotalCost waarden waren berekend volgens de logica:  *SalesQuantity * UnitCost - ReturnQuantity * UnitCost = TotalCost*.
Voor deze 20 rijen ging dit ook op. De validatie toonde echter aan dat dit voor een groot deel van de set niet geld.


*To-do*:  
- Opschonen DateKey
- Opschonen "channelkey" -> "ChannelKey"
- Controleren verband tussen SalesAmount en ReturnAmount.
- Controleren verband tussen TotalCost en ReturnAmount.

## 1.3 Data cleaning

Op basis van de analyse zien we dat de datum gegevens nu niet als juiste dtype worden ingegeven.  
In dit onderdeel worden deze omgezet naar bruikbaar datums.

In [ ]:
sales_df2 = sales_df.copy()
#Dataframe kopieeëren als backup. Zo blijft de originele dataset altijd behouden ter controle. 

sales_df2

In [ ]:
sales_df2 =sales_df2.rename(columns={"channelKey":"ChannelKey"})
#channelKey veranderen naar ChannelKey voor consistente camelcase.


In [ ]:
sales_df2["DateKey"] = pd.to_datetime(sales_df2["DateKey"], format="%Y-%m-%d %H:%M:%S,%f")
#Omzetten naar datetime van DateKey. De originele string bevat gegevens tot en met microseconden dus dient hier rekening mee gehouden worden bij de conversie.

display(sales_df2)

In [ ]:
print(sales_df2.loc[sales_df2["DateKey"].idxmin(),"DateKey"])
print(sales_df2.loc[sales_df2["DateKey"].idxmax(),"DateKey"])
#Kijken naar begin en einddatum van dataset.

De dataset loopt van 1 januari 2011 tot en met 31 december 2013

In [ ]:
datesales = sales_df2.set_index("DateKey")
#Nieuwe sales dataframe met date als index, zodat resamplen mogelijk is.

datesales.sort_values(by="DateKey")
#Sorteren van nieuwe naar oud.

In [ ]:
SalesByMonth = datesales.resample("ME").agg(TotalKeys=("SalesKey","nunique"),
                             VerkoopKanalen=("ChannelKey","nunique"),
                             TotalCost=("TotalCost","sum"),
                             SalesAmount=("SalesAmount","sum")
)
#Dataframe die kijkt naar aantal verkopen, kanalen, totale kosten en totale omzet per maand.

In [ ]:
SalesByMonth["Brutomarge"] = SalesByMonth["SalesAmount"] - SalesByMonth["TotalCost"]
#Kolom Brutomarge toevoegen aan dataframe.

SalesByMonth["BrutomargePct"] = SalesByMonth["Brutomarge"]/SalesByMonth["SalesAmount"]
#Brutomarge uitdrukken als percentage van de totale omzet.

display(SalesByMonth)
#Tussentijds inzicht in de data, niet per se nodig hier.

De Datekey is succesvol omgezet in een bruikbaar datumtype voor toekomstige analyses.
Nu dient het verband tussen SalesAmount en ReturnAmount bekeken te worden.

In [ ]:
sales_df2["SalesAmountRet"] = (sales_df2["SalesQuantity"]*sales_df2["UnitPrice"])-(sales_df2["ReturnQuantity"]*sales_df2["UnitPrice"])-sales_df["DiscountAmount"]
#Aparte kolom maken waarin returns worden verrekend met de omzet. Naast de returnquantity dient ook de discount verrekend te worden, deze werd eerder al wel correct verrekend.

display((sales_df2["SalesAmount"]-sales_df2["ReturnAmount"] == sales_df2["SalesAmountRet"]).value_counts())
#Controleren of aanname aansluit, meer dan de helft van de waarden komt terug als false.

sales_df2[~(sales_df2["SalesAmountRet"] == sales_df2["SalesAmount"]-sales_df2["ReturnAmount"])]

In [ ]:
salescontrole = sales_df2.copy()
#Copy maken van sales_df2 om te controleren of SalesAmount met returns en discounts gelijk is aan SalesAmounts met returns.
salescontrole["Check"] = salescontrole["SalesAmountRet"] - salescontrole["SalesAmount"]+ salescontrole["ReturnAmount"]
#Kolom toevoegen die controleert of SalesAmount met returns gelijk is aan SalesAmount + Returns.
display(salescontrole[salescontrole["Check"]!=0])
#Controleren op waarden die niet gelijk zijn aan 0. Deze controle geeft nog veel rijen, ook al is de controle 0, vermoeden is dat dit afrondingsverschillen zijn.
display(salescontrole[salescontrole["Check"].abs()>0.000001])
#Controleren of er waarden zijn waarvoor het voorgaande niet klopt, waarbij afrondingsverschillen van 1 decimaal worden uitgesloten door .abs()>0.000001.
#Deze bevat geen waarden dus de aanname klopt. 

In [ ]:
sales_df2["TotalCostRet"] = (sales_df2["SalesQuantity"]*sales_df2["UnitCost"]) - (sales_df2["ReturnQuantity"]*sales_df["UnitCost"])
costcontrole = np.isclose(sales_df2["TotalCostRet"],
                          sales_df2["TotalCost"],
                            atol=0.001)
#Masker om te controleren welke waarde gelijk zijn aan de berekening met een afwijking van 0.001.
display(sales_df2[~costcontrole].sort_values(by="DateKey"))
#Eerder gemaakt masker omdraaien om rijen te weergeven met waarden waar de vergelijking niet klopt. Sort_values by DateKey om te inspecteren of dit in een bepaald datumbereik zit. Het zit door heel de dataset.

Door de gehele dataset zitten waarden, waarbij de eerdere aanname van de TotalCost berekening niet aansluit, namelijk dat de returns hier al in verwerkt zaten.
Volgende aanname is: *TotalCost - (SalesQuantity * UnitCost) = TotalCostRet*.

In [ ]:
masker = np.isclose(sales_df2["TotalCostRet"],
                    sales_df2["TotalCost"]-(sales_df2["UnitCost"]*sales_df2["ReturnQuantity"]),
                      atol=0.001)
#masker om te controleren waar de aanname klopt.
display(sales_df2[~masker])
display(sales_df2.loc[masker &(sales_df2["ReturnQuantity"]>0)])

Uit de controle blijkt dat de berekeningswijze van TotalCost inconsistent is. Bij sommige transacties wordt ReturnQuantity meegenomen in de berekening van de kostprijs, terwijl dit bij andere transacties niet gebeurt.  
Deze inconsistentie kan leiden tot onjuiste uitkomsten bij verdere analyses, bijvoorbeeld bij het berekenen van de brutomarge.  
Om de kostprijs per verkoopregel op een consistente wijze vast te stellen, wordt daarom TotalCostRet gehanteerd. Deze variabele corrigeert de kostprijs voor retouren en zal in onderdeel 1.4 afzonderlijk worden gevalideerd.

### Samenvatting
De DateKey is omgezet in een datumwaarde.  
ChannelKey is renamed van "channelKey", zodat deze consistent is met de rest van de set.
Er is een kolom toegevoegd, SalesAmountRet, die omzet berekend met inachtname van retourproducten en kortingen.  
Er is een kosten kolom, TotalCostRet, toegevoegd die consequent rekening houdt met returns constante wijze is berekend.

In [ ]:
sales_df2.info()

## 1.4 Validatie
Hier worden eerdere aannames en verbanden gevalideerd. Een error op één van de onderdelen leidt tot het stoppen van het notebook, waarbij niet geëscaleerd wordt naar de volgende fase.  
Er wordt een AssertionEror met bijbehorend bericht gegenereerd waar de validatie faalt.

In [ ]:
assert sales_df2["SalesKey"].is_unique, "SalesKey bevat dubbele waarden"
# Controleert of alle SalesKeys uniek zijn.
# Bij dubbele waarden faalt de validatie met een AssertionError.

assert sales_df2["ProductKey"].notna().all(), "ProductKey bevat ontbrekende waarde"
# Controleert of iedere rij een ProductKey bevat.
# Bij ontbrekende waarden faalt de validatie met een AssertionError.


assert (sales_df2["ReturnQuantity"] <= sales_df2["SalesQuantity"]).all(), "ReturnKey is bij één of meer regels groter dan SalesQuantity"
# Controleert of ReturnQuantity de SalesQuantity niet overschrijdt.
# Wanneer dit wel voorkomt, faalt de validatie met een AssertionError.


assert np.isclose(
    sales_df2["SalesAmountRet"],
    sales_df2["SalesAmount"] - sales_df2["ReturnAmount"],
    atol=0.01
).all() , "SalesAmountRet wijkt af van SalesAmount minus ReturnAmount."
# Controleert of SalesAmountRet gelijk is aan SalesAmount minus ReturnAmount.
# np.isclose wordt gebruikt om kleine afrondingsverschillen tot € 0,01 te accepteren.


assert np.isclose(
    sales_df2["TotalCostRet"],
    sales_df2["UnitCost"] *
    (sales_df2["SalesQuantity"] - sales_df2["ReturnQuantity"]),
    atol=0.01
).all(),  "TotalCostRet wijkt af van de herberekende kostprijs na retourcorrectie."
# Controleert of TotalCostRet overeenkomt met de herberekende kostprijs
# na correctie voor retouren.

# 2 ***Analyse product data***
In dit onderdeel wordt de product data set verkend.  
Het eerste doel is om een indruk te krijgen van de inhoud en de grote van de data set.  
Ten tweede wordt er gecontroleerd op eventuele afwijkende waarden, duplicaten, en missende waarden.  
Uiteindelijk wordt er ook gekeken naar de link met de sales data set. Naar verwachting zijn er minimaal 1689 producten in de dataset.

## 2.1 Importeren dataset
De dataset wordt geimporteerd met behulp van pandas en path.

In [ ]:
product_path= Path(r"Data\Contoso_Product.csv")
#Inlezen sales.csv
product_df = pd.read_csv(product_path,
                       decimal=",")

## 2.2 Analyseren product dataset

In [ ]:
init_analyse(product_df)
#Functie aanhalen uit shared.py geschreven om te voorkoemn dat telkens dezelfde stappen doorlopen hoeven te worden.

Uit de eerste analyse blijkt dat er 1690 unique keys in de dataset zitten. Dit sluit aan op de eerdere verwachting dat er minimaal 1689 unieke keys aanwezig dienen te zijn vanuit de sales data. Er is ook geen sprake van duplicaten op basis van deze analyse. Uit de weergave blijkt dat producten in verschillende kleuren beschikbaar zijn. Dit kan later interessant zijn voor een analyse op welk kleur product het beste/slechtste verkoopt. 

Gekeken naar de datatypes zou de verwachting zijn dat de UnitCost en Unitprice als floats zouden worden ingeladen. Dit lijkt voorkomen te worden door de dollartekens in de cellen. Dit dit dient opgeschoond te worden. 

In de hele product dataset komen geen lege waarden voor.

De eerste stap is om de kosten en prijs om te zetten naar decimale waarden. Hiermee kan gecontroleerd worden of de kosten en prijs in de product dataset gelijk zijn aan die in de sales dataset.

In [ ]:
product_df["UnitCost"] = product_df["UnitCost"].str.removeprefix("$").str.replace(",",".")
product_df["UnitPrice"] = product_df["UnitPrice"].str.removeprefix("$").str.replace(",",".")
#Verwijderen van de $ prefixes uit de kolommen UnitCost en UnitPrice. Daarna de komma vervangen om floats te kunnen maken.

In [ ]:
product_df[["UnitCost","UnitPrice"]] = product_df[["UnitCost","UnitPrice"]].apply(pd.to_numeric)
#UnitCost en UnitPrice omzetten in floats.

In [ ]:
product_df.info()
#Controleren of waarden zijn omgezet.

In [ ]:
product_df.loc[~product_df["ProductKey"].isin(sales_df2["ProductKey"])]
#Zoeken naar producten wel in product dataset maar niet in de salesdata set

De A. Datum Compact Digital Camera M200 Black zit wel in de product dataset maar wordt volgens de sales data niet verkocht in de periode waar deze betrekking op heeft

In [ ]:
display(product_df.nunique())
#Aantal unieke eigenschappen onderzoeken.

In [ ]:
#Onderzoeken van strings op verschillende soorten en eventuele dubbel aanwezige gegevens met afwijkende schrijfwijzes.

display(product_df["BrandName"].value_counts())
#Onderzoeken producten van ieder merk
display(product_df["ClassName"].value_counts())
#Onderzoeken segmenten
display(product_df["Manufacturer"].value_counts())
#Onderzoeken Producenten
display(product_df["ProductDescription"].value_counts())
#Onderzoeken productomschrijvingen
display(product_df["ProductName"].value_counts())
#Onderzoeken productnamen

De verschillende merken, segmenten, producenten, productomschrijvingen en productnamen zijn geanalyseerd.
Hier komen geen dubbele waarde in voor veroorzaakt door afwijkende schrijfwijzen.   

Opvallend is het feit dat er **516** unieke omschrijvingen zijn van producten, maar **1690** unieke productbenamingen.
Dit wordt met name veroorzaakt door het feit dat iedere product meerdere kleurvarianten heeft.   

Het lijkt aan te raden om een scheiding te maken tussen productnamen en kleuren. 
Zo kan er zuiver gekeken worden naar productverkopen en later naar de voorkeur van de kleur. 
De verwachting is dat na opschonen het aantal unieke productnamen zonder kleur gelijk is aan de unieke productomschrijvingen.

De volgende stap is om te kijken of product kosten en prijzen overeenkomen met de gegevens uit de sales set.

In [ ]:
sales_df2["ProductUnitCost"] = sales_df2["ProductKey"].map(
    product_df.set_index("ProductKey")["UnitCost"]
)
#De Unitcost van de productdataset koppelen aan de sales_df2 dataset.

sales_df["CostControl"] = (
    sales_df2["ProductUnitCost"] == sales_df2["UnitCost"]
)
#Controleren of ProductUnitCost in product gelijk is aan Unitcost in de sales dataset.

In [ ]:
sales_df[~sales_df["CostControl"]]
#Controleren waar UnitCost niet overeenkomt.

In [ ]:
sales_df["ProductUnitPrice"] = sales_df["ProductKey"].map(
    product_df.set_index("ProductKey")["UnitPrice"]
)
#De UnitPrice van de productdataset koppelen aan de sales_df2 dataset.
sales_df["PriceControl"] = (
    sales_df["ProductUnitPrice"] == sales_df["UnitPrice"]
)
#Controleren of 

In [ ]:
sales_df[~sales_df["PriceControl"]]
#Onderzoeken waar prijs niet overeenkomt.

Uit vergelijkingen met de sales dataset blijkt dat kosten en prijzen voor alle sales overeenkomen met die uit de productdata.

### **Tussentijdse bevindingen**

Conform eerdere verwachtingen bevat de product dataset minimaal 1689 productkeys, het totaal is 1690. 
Het enige product dat niet voorkomt in de sales dataset is de *A. Datum Comapct Digital Camera M200 black*.  
Verder is er gekeken naar volgende eigenschappen:
- Merknamen: 10 
- Segmenten: 3 
- Producten: 10 
- Unieke productomschrijvingen: 516 
- Unieke productnamen: 1690 

Er is in deze benamingen geen besprake van duplicaten ontstaan door bijvoorbeeld verschillende schrijfwijzes.  
Het is wel opvallend dat er 516 productomschrijvingen zijn, tegenover 1690 productnamen. Nadere analyse toont aan dat dit wordt veroorzaakt door het feit dat producten verkrijgbaar zijn in meerdere kleuren. 

*To-do*:
- Aparte kolommen toevoegen voor productnaam en kleur.

## 2.3 Splitsen kolommen

In [ ]:
product_df["ProductColor"] = product_df["ProductName"].str.split().str[-1]
#Product kleuren opslaan naar ProductKolom.
product_df["ProductColor"].value_counts()
#Controleren op waarden en aantallen.

Eerst conversie laat zien dat dat er 2 verschillende wijze zijn voor blue. Eén met en één zonder hoofdletter. Daarnaast is er één waarde waarvoor de regel dat het laatste de woord de kleur is, niet op gaat. 

In [ ]:
product_df["ProductColor"]=product_df["ProductColor"].str.title()
#Schrijfwijze veranderen naar title, eerst letter is een hoofdletter, de rest kleine letters.
product_df["ProductColor"].value_counts()
#Opnieuw tellen. Enkel het probleem met de E120 blijft.
product_df.loc[product_df["ProductColor"]=="E120"]
#Er staat geen kleur  bij deze waarde, dus de standaard splits wijze gaat niet op.

In [ ]:
product_df.loc[(product_df["ProductColor"]=="E120"),"ProductColor"]="No Color"
#Waarde zoeken in color kolom en vervangen door "No Color", zodat later bekend is dat deze geen kleur heeft.
display(product_df["ProductColor"].value_counts())
display(product_df["ProductColor"].value_counts().sum())

Alle productregels hebben nu een eigen kleur kolom. 1 product heeft geen kleur.
Nu dienen de producten nog gesplitst te worden.

In [ ]:
product_df["ProductNameNoColor"] = product_df["ProductName"].str.split().str[:-1].str.join(" ")
#Splitsen productnamen van kleuren, zodat er één productnaam overblijft, op basis van de aanname dat het laatste woord de kleur betreft.
#split maakt een lijst van woorden, het laatste woord wordt verwijderd, de string wordt weer samengevoegd en zou de productnaam zonder kleur moeten weergeven.
product_df[product_df["ProductNameNoColor"].str.split().str[-1].isin(product_df["ProductColor"])]
#Controleren of laatste waarde van de productnaam zonder kleur in de kleur kolom zit.


Door de foutieve aanname date alle kleuren 1 woord zijn is silver Grey nu niet goed in de kleur kolom gezet. Dit dient gecorrigeerd te worden alvorens verder te gaan. 

In [ ]:
product_df.loc[product_df["ProductName"].str.contains("Silver Grey"),"ProductColor"] = "Silver Grey"
#Zoek de productnamen met "Silver Grey" en zet de waarde voor ProductColor naar Silver Grey.

In [ ]:
kleuren = list(product_df["ProductColor"].unique())
#maak een list van kleuren.
masker = product_df["ProductName"].apply(lambda naam: any(kleur in naam for kleur in kleuren))
#Masker dat controleert of kleur in de productnaam voor komt.
display(product_df[~masker])
#Laat de producten zien waarvoor dit niet van toepassing is. Dit zijn de producten die we eerder gecorrigeerd hebben.


Kleur lijkt nu opgelost te zijn. Nu een zuivere productnaam creeëren.

In [ ]:
product_df["ProductNameNoColor"] = product_df.apply(
    lambda row: row["ProductName"].removesuffix(row["ProductColor"]).strip(),
    axis=1
)
#Nieuwe functie die met behulp van apply voor iedere rij een nieuwe waarde berekent die de productcolor verwijdert van de originele productname.


In [ ]:
product_df["ProductNameNoColor"] = product_df.apply(
    # Voert de onderstaande lambda-functie uit op iedere rij van product_df.
    lambda row: re.sub(
        # Vervangt tekst op basis van een regex-patroon.
        rf"{re.escape(row['ProductColor'])}$",
        # Bouwt het zoekpatroon op uit de kleur van de huidige rij.
        # re.escape() zorgt dat eventuele speciale regextekens in de kleur veilig worden behandeld.
        # $ betekent: de kleur moet aan het einde van ProductName staan.
        "",
        # Vervangt de gevonden kleur door een lege string.
        row["ProductName"],
        # Dit is de tekst waarin gezocht en vervangen wordt.
        flags=re.IGNORECASE
        # Negeert hoofdletters en kleine letters.
        # Dus "Blue", "blue" en "BLUE" worden allemaal gevonden.
    ).strip(),
    # Verwijdert eventuele overgebleven spaties aan het begin of einde.
    axis=1
    # Geeft aan dat apply rij voor rij moet werken in plaats van kolom voor kolom.
)

In [ ]:
Duplicatecontrole = product_df[["ProductNameNoColor","ProductDescription"]].drop_duplicates()
#Dataframe makenn met alleen productnamen en descriptions.
Duplicatecontrole["ProductDescription"].value_counts()
#Aantal keren tellen dat waarden voorkomen. 


De dataframe met alleen productnames en descriptions geeft 516 unieke descriptions met maximaal 20 keer hergebruikt en minimaal 1 keer.
Dit bevestigt dus dat er voor meerdere producten dezelfde description wordt gebruikt.  

In [ ]:
product_df

In [ ]:
#Masker maken ter controle van de kleuren in de dataset.
masker_color = pd.Series(
    [
        pd.notna(kleur)
        #Controleert of waarden in ProductColor gevuld is.
        and pd.notna(naam)
        # Controleert of ProductNameNoColor gevuld is.
        and bool(
            re.search(
                rf"\b{re.escape(str(kleur))}\b",
                str(naam),
                flags=re.IGNORECASE
            )
        )
        #Controleert of ProductColor in ProductNameNoColor voorkomt. De regex-expressie met \b voorkomt dat Blue in Bluetooth een false positive geeft.
        for naam, kleur in zip(
            product_df["ProductNameNoColor"],
            product_df["ProductColor"]
        )
        #Geeft aan voor welke waarden dit gecontroleerd moet worden. 
        #Zip vormt een iterator die per iteratie een combinatie van ProductNameNoColor en ProductColor uit dezelfde rij oplevert.
    ],
    index=product_df.index
    #Index van dataframe toepassen op masker.
)

product_df[masker_color]
#Laat rijen zien, waarvoor de kleur nog wel in ProductNameNoColor voorkomt.

In [ ]:
product_df

## 2.4 Validatie

In [ ]:
assert product_df["ProductKey"].is_unique,  "ProductKey bevat één of meerdere duplicaten."
#Controleert of alle ProductKeys uniek zijn.

assert product_df["ProductKey"].notna().all(), "ProductKey bevat één of meerdere ontbrekende waarden."
#Controleert of er waarden ontbreken in de ProductKey.

assert not masker_color.any(), "Eén of meerdere productkleuren worden nog benoemd in de productnaam zonder kleur."
#Controleert of er kleuren voorkomen in de ProductNoColor kolom.

# 3 ***Analyse calendar data***
In dit onderdeel kijken we naar de calendar dataset.
Naar verwachting zal deze matchen op de datekey en minimaal datums van 1 januari 2011 tot en met 31 december 2013 bevatten.

### 3.1 Importeren dataset
Met behulp van path en pandas importeren wordt de dataframe geïmporteerd voor analyse.

In [ ]:
calendar_path= Path(r"Data\Contoso_calendar.csv")
#Inlezen calendar.csv als raw string om speciale tekens te escapen.
calendar_df = pd.read_csv(calendar_path,
                       decimal=",")

### 3.2 Analyse dataset
Hier voeren we de eerste analyse uit op de calendar dataset.

In [ ]:
init_analyse(calendar_df)
#Met behulp van analyse functie eerste algemene gegeens opvragen.

De dataset is groter dan de sales dataset qua datumbereik. Dit is opzich geen probleem, echter kan dit wel voor extra load zorgen gezien gegevens die niet worden gebruikt worden ingeladen.
Een actiepunt is om eventueel later te kijken naar het schrappen van datums die niet relevant zijn om performance te verbeteren.
Alle waarden zijn gevuld en volgen een standaard structuur. 
De Year kolom lijkt 2 keer aanwezig, eventueel kan er één gedropt worden uit de set. Daarnaast lijken Day en DayOfMonth dezelfde gegevens te weergeven. 

Op het moment is het niet zeker of deze tabel in PowerBI direct werkt, voor pandas gebruik hebben we een kolom nodig die als datetime functioneert. 
De volgende stap is om op voorhand een kolom te maken die als datetime gebruikt kan worden. 

In [ ]:
calendar_df["DateInt"].isin(sales_df["DateKey"])
#Controleren matches in sales dataset.

In de sales data wordt gebruik gemaakt van een DateKey kolom. Deze waarde komt niet terug in de calendar dataset en matcht ook niet op DateInt.
De DateInt dient nog als Key aangemaakt te worden als waarde in de sales set.

### 3.3 Data cleaning
In dit onderdeel maken we met de aanwezige gegevens een DateKey om te matchen op de waarde in de sales set.

In [ ]:
calendar_df.head(20)

In [ ]:
calendar_df["DateKey"] = pd.to_datetime(calendar_df["DateInt"], format="%Y%m%d")
#Datekey aanmaken in calendar set door de DateInt te converteren naar een datetime op basis van het format.


In [ ]:
calendar_df[calendar_df["DateKey"].isin(sales_df2["DateKey"])]
#Controleren of de DateKey matcht op waarden in de sales data. 

Met behulp van de DateIn waarde is een DateKey aangemaakt die matcht op gegevens in de sales dataset. 
Bij Controle van de dataset start deze op 1 januari 2011 en eindigt deze op 31 december 2013. Dit komt overeen met de sales dataset.
Dit lijkt voornamelijk relevant voor het PowerBI gedeelte.

In [ ]:
sales_df2["DateInt"] = sales_df2["DateKey"].dt.strftime("%Y%m%d").astype(int)
#Waarden van Datekey veranderen naar integer door middel van strftime en astype. Dit is nodig om de DateKey te kunnen matchen met de DateInt in de calendar dataset.

In [ ]:
sales_df2["DateInt"].isin(calendar_df["DateInt"]).value_counts()
#Controleren of er waarden zijn die niet voorkomen in calendar.

In [ ]:
sales_df2 = sales_df2.rename(columns={"DateKey":"Date",
                          "DateInt":"DateKey"})
#Columns renamen, zodat key koppelt op DateKey in calendar.

In [ ]:
calendar_df = calendar_df.rename(columns={"DateKey":"Date"})

In [ ]:
calendar_df = calendar_df.rename(columns={"DateInt":"DateKey"})

In [ ]:
display(calendar_df)

## **Voorlopige conclusie**

De calendar data lijkt met name voor de tijdfunctie binnen PowerBI.
Voor de zekerheid is een key aangemaakt om op te matchen met de sales data.  
Day en DayOfMonth lijken dezelfde waarde te bevatten, idem voor Year en Year.1

*actiepunten*:
- Eventueel dubbele waarden voor day en year laten vallen.
- Tabel verkleinen tot bereik van de dataset.

In [ ]:
calendar_df = calendar_df.drop(["Day","Year.1"],axis=1)

Dubbele waarden zijn verwijderd en er is een werkende datekey gemaakt.

In [ ]:
calendar_df2 = calendar_df[calendar_df["Date"].between("01-01-2011","31-12-2014")].reset_index()
#Een nieuwe calendar dataset met alleen datums relevant tot de dataset om performance optimaal te houden.

## 3.4 Validatie

In [ ]:
assert calendar_df2["DateKey"].is_unique, "DateKey bevat één of meerdere dubbele datums."
#Controleert of alle DateKeys uniek zijn.

assert calendar_df2["DateKey"].notna().all(), "DateKey bevat één of meerdere ontbrekende waarden."
#Controleert of er lege waarden in de DateKey aanwezig zijn.

assert pd.api.types.is_datetime64_any_dtype(calendar_df2["Date"]), "DateKey is niet omgezet naar een geldig datetime datatype."
#Controleert of DateKey correct is omgezet naar een datumwaarde.

# 4 ***Analyse geography data***
In dit onderdeel wordt de product dataset verkend.  
Het eerste doel is om een indruk te krijgen van de inhoud en de grote van de data set.  
Ten tweede wordt er gecontroleerd op eventuele afwijkende waarden, duplicaten, en missende waarden.  
Uiteindelijk wordt er ook gekeken naar de link met de sales data set. 

### 4.1 Importeren dataset


In [ ]:
geography_path= Path(r"Data\Contoso_Geography.csv")
#Inlezen geography.csv
geography_df = pd.read_csv(geography_path,
                       decimal=",")

### 4.2 Analyse dataset

In [ ]:
init_analyse(geography_df)

In [ ]:
geography_df["GeographyType"].value_counts()

De dataset lijkt opgebouwd uit continenten, landen, provincies/regios en steden.  
Bij regioncountryname zijn er 3 lege waarden gevonden in de initiële analyse. Hier dient op ingezoomd te worden. Er is geen sprake van duplicated.

In [ ]:
geography_df[geography_df["RegionCountryName"].isna()]
#Waarden met ontbrekende regioncountryname bekijken.

De lege waarden lijken betrekking hebben op contintent.
Deze wordt ook al in ContinentName benoemd. 
Het lijkt dus logisch om deze waarden te verwijderen uit de dataset om de data duidelijke te presenteren.

In [ ]:
geographystr = ["ContinentName","GeographyType","RegionCountryName"]

for g in geographystr:
    display(geography_df[g].value_counts())
#Onderzoeken dataset op afwijkende schrijfwijzes, daarnaast de samenstelling van de geography set.

In [ ]:
# GeographyKey moet uniek zijn
display(geography_df["GeographyKey"].is_unique)

# Verdeling over hiërarchieniveaus
display(geography_df["GeographyType"].value_counts())

# Zelfde naam op verschillende niveaus bekijken
display(geography_df[
    geography_df["RegionCountryName"].eq("the Netherlands")
])

## 4.3 Data Cleaning

In [ ]:
geography_df2 = geography_df.loc[~(geography_df["RegionCountryName"].isna())]
#Maakt nieuwe dataframe waarin continents uit geographytype worden gedropt voor duidelijkere analyses.

Contintent gedropt uit de data set, gezien deze ook al benoemd wordt in de kolom "ContinentName".

## 4.4 Validatie

In [ ]:
assert geography_df2["GeographyKey"].is_unique, "GeographyKey bevat één of meerdere duplicaten."
# Controleert of alle GeographyKeys uniek zijn.


assert geography_df2["GeographyKey"].notna().all(),  "GeographyKey bevat één of meerdere ontbrekende waarden."
# Controleert of iedere rij een GeographyKey bevat.


assert geography_df2["RegionCountryName"].notna().all(), "RegionCountryName bevat één of meerdere ontbrekende waarden."
# Controleert of de continentregels succesvol zijn verwijderd en iedere overgebleven rij een RegionCountryName bevat.


assert geography_df2["ContinentName"].notna().all(), "ContinentName bevat één of meerdere ontbrekende waarden."
# Controleert of ContinentName in alle kolommen is ingevuld.

## Voorlopige conclusie
De geography dataset bevate gegevens omtrent locaties. Deze kunnen niet direct gelinkt worden aan de sales dataset.  
De voorlopige aanname is dat deze samenhangt met store data.
De gegevens hebben met name betrekking op de Verenigde Staten, daarop volgt Europa en uiteindelijk Europa.

Het zou interessant kunnen zijn om ontwikkelingen van de verkopen op deze verschillende contintenten te zien en vergelijken. 
Eventueel product context.

# 5 ***Analyse store data***
In dit onderdeel wordt de store dataset verkend.  
Het eerste doel is om een indruk te krijgen van de inhoud en de grote van de data set.  
Ten tweede wordt er gecontroleerd op eventuele afwijkende waarden, duplicaten, en missende waarden.  
Uiteindelijk wordt er ook gekeken naar de link met de sales dataset en de geography dataset

### 5.1 Data importeren

In [ ]:
stores_path= Path(r"Data\Contoso_Stores.csv")
#Inlezen stores.csv
stores_df = pd.read_csv(stores_path,
                       decimal=",")

### 5.2 Data analyseren

In [ ]:
init_analyse(stores_df)

Volgens de dataset zijn er 306 unieke storekeys dit komt overeen met eerdere analyse uit de sales data.  
Met name in de CloseReason kolom zijn veel lege waarden. De aanname is dat deze pas wordt ingevuld bij een sluiting van een locatie. Er is ook geen rekening gehouden met SCD. Een tabel die registreert wanneer de CloseReason wijzigt zou meer info kunnen geven over deze wijzigingen.  
Er is geen data aanwezig over wanneer de winkel is gesloten, het lijkt dus niet mogelijk te oordelen of er foutieve toewijzigingen aan een winkel zijn in de salessset.
Storetypes lijkt samen te kunnen hangen met de verkoopkanalen die voort komen uit de sales dataset.

Daarnaast ontbreekt er bij 1 store een employeecount, hier dient naar gekeken te worden. Tevens worden employeecounts op het moment als floats opgeslagen, ervanuit gaande dat dit om personen gaat en niet FTE.

Zoals eerder verwacht is de geographykey verwerkt in deze dataset.

In [ ]:
display(stores_df[stores_df["EmployeeCount"].isna()])

Betreft store met ontbrekende EmployeeCount. 

In [ ]:
display(sales_df2.loc[(sales_df2["StoreKey"] == 210),["SalesKey","Date","SalesAmountRet"]].sort_values(by="Date").resample("QE", on="Date").agg(aantal_sales=("SalesKey","count"),
                                                                                                                                                         Omzet=("SalesAmountRet","sum")))

Lijkt op een winkel met teruglopende omzet, wellicht daarom geen werknemers?

In [ ]:
per_store_kwartaal = (
    sales_df2
    .groupby([
        "StoreKey",
        pd.Grouper(key="Date", freq="QE")
    ])
    .agg(
        aantal_sales=("SalesKey", "count"),
        omzet=("SalesAmountRet", "sum")
    )
)

per_store_kwartaal

gemiddelde_per_kwartaal = (
    per_store_kwartaal
    .groupby("Date")
    .agg(
        gemiddeld_aantal_sales=("aantal_sales", "mean"),
        mediaan_aantal_sales=("aantal_sales", "median"),
        gemiddelde_omzet=("omzet", "mean"),
        mediaan_omzet=("omzet", "median")
    )
)

gemiddelde_per_kwartaal

store_210 = per_store_kwartaal.loc[210]

vergelijking = store_210.join(
    gemiddelde_per_kwartaal
)

vergelijking

Winkel blijft qua verkopen en omzet achter op gemiddelde en mediaan. Echter geen conclusie vast kunnen maken aan reden dat er geen employee count is ingegeven. 
Eventueel rij droppen als deze problemen geeft.

In [ ]:
Control = ["CloseReason","Status","StoreName","StoreType"]
display(stores_df.nunique())

for c in Control:
    display(stores_df[c].value_counts())
#Loop om values te weergeven bij unieke waardes.



Volgende feiten vallen op gekeken naar de verschillende omschrijvingen en namen:
- 2 sluitingsreden
- 263 geographykeys t.o.v. 674 in de geography set
- 15 selling area sizes, de vraag is of dit daadwerkelijke oppervlakte is of een soort bucket?
- 4 storetypes, lijkt dus gelijk te zijn aan de verkoopkanalen.

 

In [ ]:
stores_df["StoreKey"].isin(sales_df2["StoreKey"])
sales_df2["StoreKey"].isin(stores_df["StoreKey"])
#Checken of alle waarden aanwezig zijn in salesset

In [ ]:
display(geography_df2[~geography_df2["GeographyKey"].isin(stores_df["GeographyKey"])])
#GeographyKeys niet in de store dataset.

display(geography_df2[(geography_df2["GeographyKey"].isin(stores_df["GeographyKey"]))])
#GeographyKeys in de store set.

display(geography_df2.loc[(geography_df2["GeographyKey"].isin(stores_df["GeographyKey"]))&(geography_df2["GeographyType"]!="City")])
#GeographyKeys in de store set, waarvan het GeographyType niet City is.

## Voorlopige Conclusie
Storekeys in de salesset komen overeen met de sales set. 
Een groot aantal geography locaties komen niet voor in de storeset. 
Dit komt name doordat er zowel waarden aanwezig zijn voor Country/region, province en city. Deze worden dus op meerdere lagen meegenomen.
Verdere analyse laat zien dat de storedataset alleen informatie op city niveau bevat. De Continentname en RegioCountryname kolommen geven hier invulling aan het continent en het land, over de provincie en city zijn geen verdere informatie in de dataset aanwezig. 

Er zijn geen historische gegevens beschikbaar met betrekking tot de kolom CloseReason. Daarnaast ontbreken definities van de waarden "Store" en "Relocation".

Wellicht interessant om in kaart te brengen of er locaties/regio's zijn waar veel verkopen zijn via andere kanalen waar een store van toegevoegde waarde zou zijn.
Het lijkt logischer de GeographyKey direct aan de sales te verbinden gezien dit een dimensie van de verkoop is

*To-do*:
- GeographyKey toevoegen aan sales voor directe link.


## 5.3 Data cleaning

In [ ]:
mergeid = sales_df2.copy()
#Sales_df 2 kopieeren naar een nieuwe variabele te testen of de mapping werkt.
mergeid["GeographyKey"] = mergeid["StoreKey"].map(stores_df.set_index("StoreKey")["GeographyKey"])
#GeographyKey toevoegen door middel van  .map functie. Als de StoreKey in mergeid wordt gevonden, wordt de bijbehorende GeographyKey uit de storedataset toegevoegd.
sales_df2 = mergeid
#Nieuwe sales data set met geographykey wordt leidend.
display(geography_df2["GeographyKey"].isin(sales_df2["GeographyKey"]).value_counts())
#Controleren of aantal geographykeys in sales overeenkomt met de eerdere controle, hoeveel geographykeys aanwezig zijn in de dataset.

De GeographyKey is direct gelinkt aan de salesdata gezet, omdat dit een dimensie van sales is.  
Controle uitgevoerd door te controleren of aantal GeographyKeys uit de Geography dataset aansluit met de waarde die de vergelijking tussen stores en geography eerder gaf.  
Er is voor gekozen de GeographyKey ook in de storeset te behouden, zodat er ook nog specifieke data over winkels opgevraagd kan worden.

## 5.4 Validatie


In [ ]:
assert stores_df["StoreKey"].is_unique,  "StoreKey bevat één of meerdere duplicaten."
#Controleert of er duplicaten in de StoreKey aanwezig zijn.

assert stores_df["StoreKey"].notna().all(),  "StoreKey bevat één of meerdere ontbrekende waarden."
#Controleert of er rijen zijn waarbij de StoreKey ontbreekt.

assert stores_df.loc[stores_df["Status"] == "On","CloseReason"].isna().all(), "Eén of meerdere actieve winkels hebben een CloseReason."
#Controleert of er actieve winkels zijn met een CloseReason.

assert stores_df.loc[stores_df["Status"] == "Off","CloseReason"].notna().all(), "Eén of meerdere gesloten winkels hebben geen CloseReason."
#Controleert of er gesloten winkels zijn zonder CloseReason.

# 6 ***Analyse Promotions data***
In dit onderdeel wordt de promotions dataset verkend.  
Het eerste doel is om een indruk te krijgen van de inhoud en de grote van de data set.  
Ten tweede wordt er gecontroleerd op eventuele afwijkende waarden, duplicaten, en missende waarden.  
Uiteindelijk wordt er ook gekeken naar de link met de sales data.

### 6.1 Data importeren

In [ ]:
promotions_path= Path(r"Data\Contoso_Promotion.csv")
#Inlezen promotions.csv
promotions_df = pd.read_csv(promotions_path)
#Decimal weggehaald, gezien set met . werkt voor decimalen.


### 6.2 Data analyse

In [ ]:
init_analyse(promotions_df)

DiscountPercent is in tegenstelling tot voorgaande datasets met een . voor decimalen aangegeven. 
Er zijn 2 datum waarden aanwezig in de set. Een eerste analyse lijkt er op te wijzen dat dit een begin en eind datum van de promotie is, waarbij de datums eindigend op .1 de startdatums zijn. 
Door deze om te zetten naar datums kunnen we dit later gebruiken om te analyseren hoeveel er gebruik gemaakt is van de promoties in de periode.
Eventueel vergelijkingen maken tussen periodes met en zonder promotie en de winstgevendheid hiervan.  
Het aantal promotionkeys komt overeen met het aantal in de sales set. Promotionlabel lijkt altijd gelijk te zijn aan de key, onduidelijk waarom.  
Er zitten ook waarden in de dataset met betrekking tot 2014, deze kunnen niet gerelateerd worden aan de salesdata gezien deze loopt tot 31 december 2023.
Promotion names lijken grotendeels per jaar overeen te komen. Effectiviteit over de afgelopen 3 jaar zou dus bekeken kunnen worden. 
Eventueel waarde "European Back-to-Scholl Promotion" opschonen naar School.

Key 1 lijkt een doorlopende promotie te zijn om verkopen met geen korting vast te leggen.
Is er correlatie tussen promotion en discountamount? 
De volgende stap is om de samenhang met de salesdata te analyseren. Er is hier een aansluiting op de SalesKey.



In [ ]:
display(promotions_df["PromotionKey"].isin(datesales["PromotionKey"]).value_counts())
display(datesales["PromotionKey"].isin(promotions_df["PromotionKey"]).value_counts())
#Controleren welke promotionkeys in de salesset zitten en vice versa.


In [ ]:
datesales.loc[datesales["PromotionKey"] == 1, "DiscountAmount"].sum()
#Controleren of alle gegevens met promotionkey 1 geen discount hebben. 

In [ ]:
promo2 = datesales.loc[datesales["PromotionKey"]==2].sort_index()
#Alle waarden voor promo 2 opzoeken en sorteren op index en in aparte dataframe zetten.

In [ ]:
#Controleren discountamount op basis van promotionkey 2.
#Verwachting DiscountPct is 5% van aanschafwaarde
promo2["SalesBeforeDiscount"] = promo2["UnitPrice"]*promo2["SalesQuantity"]
promo2["DiscountPct"] = promo2["DiscountAmount"]/promo2["SalesBeforeDiscount"]*100
#Geen directe aansluiting tussen percentage en discountamount.
#Vermoeden UnitPrice*DiscountQuantity* DiscountPercent = DiscountAmount
promo2["DiscountPercent"] = promo2["PromotionKey"].map(promotions_df.set_index("PromotionKey")["DiscountPercent"])
#Discountpercentage koppelen aan waarde in promo dataframe. Zoek promotion key in promo en match deze op promotion dataset, voegt verolgens de discountpercent toe als kolom.
promo2["DiscountControle"] = promo2["UnitPrice"]*promo2["DiscountQuantity"]*promo2["DiscountPercent"]
#Discount herrekenen ter controle
promo2a = promo2.loc[(promo2["DiscountControle"].round(2) != promo2["DiscountAmount"].round(2)),["DiscountAmount","DiscountControle"]]
promo2a["ControleDiscountAmount"] = promo2a["DiscountAmount"] - promo2a["DiscountControle"]
promo2a.head(50)
promo2a[(promo2a["ControleDiscountAmount"]<-0.001)|(promo2a["ControleDiscountAmount"]>0.001)]
#Controleren afwijkingen, lijkt alleen op afrondingen terecht te komen.

Op basis van PromotionKey 2 is het verband tussen de PromotionKey het discountpercentage in de promotions dataset en de DiscountAmount in de sales data set geanalyseerd.  
De UnitPrice * DiscountQuantity * DiscountPercent is gelijk aan DiscountAmount voor deze promocode.  
Op basis hiervan lijken er specifieke voorwaarden aan promotieacties te zitten over de hoeveelheden waarop korting wordt geleverd.  
De volgende actie is om deze controle voor alle promotioncodes uit te voeren in de salesdataset.

In [ ]:
promocontrole = datesales.copy()
#Kopieeren van salesdataset
promocontrole["DiscountPercent"] = promocontrole["PromotionKey"].map(promotions_df.set_index("PromotionKey")["DiscountPercent"])
#Mappen van discountpercentage
promocontrole["DiscountControle"] = promocontrole["UnitPrice"]*promocontrole["DiscountQuantity"]*promocontrole["DiscountPercent"]
#Berekeken discountamount
promocontrole["ControleDiscountAmount"] = promocontrole["DiscountAmount"]-promocontrole["DiscountControle"]
display(promocontrole[promocontrole["ControleDiscountAmount"] != 0])
#Controleren waarden niet gelijk aan 0
display(promocontrole[(promocontrole["ControleDiscountAmount"]<-0.001)|(promocontrole["ControleDiscountAmount"]>0.001)])
#Controleren afrondingsverschillen.

Nu testen we dit voor de gehele dataset.

In [ ]:
datesales["DiscountPercent"] = datesales["PromotionKey"].map(promotions_df.set_index("PromotionKey")["DiscountPercent"])
datesales["DiscountControle"] = datesales["UnitPrice"]*datesales["DiscountQuantity"]*datesales["DiscountPercent"]
datesales["ControleDiscountAmount"] = datesales["DiscountAmount"]-datesales["DiscountControle"]
#Herhalen stappen voor controle

display(datesales[datesales["ControleDiscountAmount"] != 0])
#Dataframe weergeven waar controle waarde niet gelijk is aan 0.
display(datesales[(datesales["ControleDiscountAmount"]<-0.001)|(datesales["ControleDiscountAmount"]>0.001)])
#Dataframe met waardes met een afwijking groter dan een duizendste achter de decimaal.

Bovenstaande controle bevestigt het vermoeden dat UnitPrice * DiscountQuantity * DiscountPercent gelijk is aan DiscountAmount.
Er lijkt dus een verband te zijn tussen promotion en de hoeveelheid items. 
Promotions kunnen bijvoorbeeld zijn vanaf een bepaalde hoeveelheid of bij een bepaalde hoeveelheid een product gratis. Het is niet helemaal duidelijk voor iedere promotie wat deze verbanden zijn en of deze relevant zijn.

In [ ]:
datesales.loc[(datesales["PromotionKey"]==2)&(datesales["DiscountAmount"]>0)].sort_index().head(20)
#Kijken naar eerste rijen met DiscountQuantity en PromotionKey 2.
datesales.loc[(datesales["PromotionKey"]==2)&(datesales["DiscountAmount"]>0)].sort_index().tail(20)
#Laate 50 waarden


In [ ]:
datesales.loc[datesales["PromotionKey"]==2].groupby("SalesQuantity")["DiscountQuantity"].describe()
#Kijken naar ratio's verkoop hoeveelheid afgezet tegen hoeveelheid discounts.

Voor nu lijkt er verder geen logisch verband te zijn tussen promotion acties en wanneer deze wel of niet wordt gegeven en op welke hoeveelheid producten.

## Voorlopige Conclusie
De promotieset bevat data over 28 promoties die open vanaf 1 januari 2011 tot en met 31 maart 2014.
De Promotion met Key 1 lijkt de standaardwaarde te zijn voor verkopen zonder enige korting.
Het verband met de salesset is het feit dat UnitPrice * DiscountQuantity * DiscountPercent gelijk is aan de DiscountAmount.

Er is geen duidelijk verband te vinden tussen de verschillende promotieacties en hoeveelheden. De verwachting zou zijn dat kortingen gelden vanaf een aantal items of per aantal items worden toegepast. Dit verband is echter niet te herleiden.

**To-do**:
- De set heeft 2 datumwaarden. De eerste heeft betrekking op de einddatum de tweede op de begindatum van de promotion. Deze dienen nog duidelijk benoemd te worden. 
- Er zit een fout in de omschrijving van de Europese back-to-scholl actie. Deze nog aanpassen naar "European Back-to-School Promotion".

## 6.3 Data cleaning
Aan de hand van de to-do list wordt de data opgeschoond.

In [ ]:
promotions_df["EndMonth"] = pd.to_datetime(promotions_df["Month"],format="%B").dt.month.astype("int64")
#Met behulp van datetimeseries de maandnamen omzetten naar integers om later de datum mee te berekenen.
kolom = promotions_df.pop("EndMonth")
promotions_df.insert(3,"EndMonth", kolom)
#Kolom verplaatsen naar begin met rest van data.
promotions_df.info()

In [ ]:
promotions_df["PromotionEndDate"] = pd.to_datetime({"year":promotions_df["Year"],"month":promotions_df["EndMonth"],"day":promotions_df["Day"]})
#Op basis van jaar, maand en, dag gegevens einddatum als datumwaarde aanmaken in de set.

In [ ]:
promotions_df["StartMonth"] = pd.to_datetime(promotions_df["Month.1"],format="%B").dt.month.astype("int64")
#Startmaand converteren naar integer
promotions_df["PromotionStartDate"] = pd.to_datetime({"year":promotions_df["Year.1"],"month":promotions_df["StartMonth"],"day":promotions_df["Day.1"]})
#Op basis van jaar, maand en, dag gegevens einddatum als datumwaarde aanmaken in de set.


In [ ]:
promotions_df = promotions_df.rename(columns={"Year": "EndYear",
                              "Month": "EndMonthstr",
                              "Quarter": "EndQuarter",
                              "Day": "EndDay",
                              "Year.1":"StartYear",
                              "Quarter.1":"StartQuarter",
                              "Month.1":"StartMonthstr",
                              "Day.1":"StartDay"})
#Kolomnamen hernoemen naar duidelijk benamingen die eind en begin aangeven.

In [ ]:
promotions_df.loc[promotions_df["PromotionName"]== "European Back-to-Scholl Promotion","PromotionName"] = "European Back-to-School Promotion"
#Corrigeren foutieve schrijfwize European Back-to-School Promotion


In [ ]:
kolom = promotions_df.pop("StartMonth")
promotions_df.insert(11,"StartMonth", kolom)
#Verplaatsen StartMonth

De eind-, en startdatumwaarde zijn duidelijk benoemd binnen de dataset.
Deze zijn ook geconverteerd tot bruikbare datevalues. Deze maken ook duidelijk dat de laatste promotion met 0% op het moment tot en met  eind 2014 loopt. 
De datum set zal hier op aangepast worden. 
Verder is de foutieve schrijfwijze van de European Back-to-School promotion aangepast.

## 6.4 Validatie

In [ ]:
assert promotions_df["PromotionKey"].is_unique, "PromotionKey bevat één of meerdere duplicaten."
# Controleert of alle PromotionKeys uniek zijn.

assert promotions_df["PromotionKey"].notna().all(), "PromotionKey bevat één of meerdere ontbrekende waarden."
# Controleert of iedere promotie een PromotionKey bevat.

assert promotions_df["DiscountPercent"].between(0, 1).all(), "DiscountPercent bevat één of meerdere waarden buiten het bereik van 0 tot 1."
# Controleert of alle kortingspercentages tussen 0% en 100% liggen.

assert pd.api.types.is_datetime64_any_dtype(promotions_df["PromotionStartDate"]), "PromotionStartDate is niet omgezet naar een geldig datetime datatype."
# Controleert of PromotionStartDate correct is omgezet naar een datumwaarde.

assert pd.api.types.is_datetime64_any_dtype(promotions_df["PromotionEndDate"]), "PromotionEndDate is niet omgezet naar een geldig datetime datatype."
# Controleert of PromotionEndDate correct is omgezet naar een datumwaarde.

assert (promotions_df["PromotionStartDate"]<= promotions_df["PromotionEndDate"]).all(), "Eén of meerdere promoties hebben een startdatum na de einddatum."
# Controleert of iedere promotie begint voor of op de bijbehorende einddatum.

# 7 ***Analyse Product (Sub)Category***

In [ ]:
productcat_path= Path(r"Data\Contoso_Product_Category.csv")
#Inlezen product categories.csv
productcat_df = pd.read_csv(productcat_path)
#Decimal weggehaald, gezien set met . werkt voor decimalen.

In [ ]:
init_analyse(productcat_df)

Dataset lijkt een subcategory te zijn van een andere dataset. 
Vermoedelijk de Product Sub Category dataset.
Verbinding lijkt te zijn ProductKey -> ProductSubCategroy -> ProductCategory
Eerst product Sub Category bekijken.

In [ ]:
productsubcat_path= Path(r"Data\Contoso_ProductSubCategory.csv")
#Inlezen product subcategories.csv
productsubcat_df = pd.read_csv(productsubcat_path)
#Decimal weggehaald, gezien set met . werkt voor decimalen.

In [ ]:
init_analyse(productsubcat_df)

Er lijken 44 unieke Subcategorieën te zijn, verbonden aan 1 van de 8 hoofdcategorieën.
Volgende stap is om te kijken hoeveel van de subcategorykeys zich in de productlijst bevinden.

De header ProductSubcategory is niet in consistente camelcase

In [ ]:
masker = productsubcat_df["ProductSubcategoryKey"].isin(product_df["ProductSubcategoryKey"])
#filter die controleert welke keys wel of niet in de product dataset zitten.
print(f"{productsubcat_df.loc[masker,["ProductSubcategoryKey"]].nunique().sum()} Productsubcategorieën in product dataframe")
display(productsubcat_df[masker])
#Geeft alle waarden weer die wel in de dataset zitten.
print(f"{productsubcat_df.loc[~masker,["ProductSubcategoryKey"]].nunique().sum()} Productsubcategorieën niet in product dataframe")
display(productsubcat_df[~masker])

Dataset lijkt met namen hardware en multimedia apparaten te betreffen.  De 22 waarden sluiten ook aan met de unique count van de productset.

De productgroep die niet in de productcategory voorkomt heet met name berekking op witgoed, media en audio.

In [ ]:
productsubcat_df["ProductCategoryKey"].isin(productcat_df["ProductCategoryKey"])
productcat_df["ProductCategoryKey"].isin(productsubcat_df["ProductCategoryKey"])
#Controleren of alle keys in de productcategoryKey zitten.

In [ ]:
print("Categorieën in product data set:")
display(productcat_df[(productcat_df["ProductCategoryKey"]).isin(productsubcat_df.loc[masker,"ProductCategoryKey"])])
#Controleren welke productcategorieën voorkomen in products
print("Categorieën niet in product data set:")
productcat_df[~(productcat_df["ProductCategoryKey"]).isin(productsubcat_df.loc[masker,"ProductCategoryKey"])]
#Controleren welke productcategorieën niet voorkomen in products

Conform de hypothese zijn witgoed en media uitgesloten van de set.

## Voorlopige conclusie

De dataset lijkt 22 van de 44 subcategorieën te gebruiken.
Daarnaast zijn de categoriën met ProductCategoryKey 7 en 8 niet van toepassing op de sales dataset.

*To-do*
- Productcategory en subcategory mergen voor snellere toegang.

## 7.3 Data cleaning

In [ ]:
productmerge_df = productsubcat_df.merge(productcat_df, on="ProductCategoryKey")

## 7.4 Validatie

In [ ]:
assert productcat_df["ProductCategoryKey"].is_unique, "ProductCategoryKey bevat één of meerdere duplicaten."
# Controleert of alle ProductCategoryKeys uniek zijn.

assert productcat_df["ProductCategoryKey"].notna().all(), "ProductCategoryKey bevat één of meerdere ontbrekende waarden."
# Controleert of iedere productcategorie een ProductCategoryKey bevat.

assert productsubcat_df["ProductSubcategoryKey"].is_unique, "ProductSubcategoryKey bevat één of meerdere duplicaten."
# Controleert of alle ProductSubcategoryKeys uniek zijn.

assert productsubcat_df["ProductSubcategoryKey"].notna().all(),  "ProductSubcategoryKey bevat één of meerdere ontbrekende waarden."
# Controleert of iedere productsubcategorie een ProductSubcategoryKey bevat.

assert productsubcat_df["ProductCategoryKey"].notna().all(), "ProductCategoryKey bevat één of meerdere ontbrekende waarden in de subcategorie dataset."
# Controleert of iedere subcategorie aan een productcategorie gekoppeld is.

assert productsubcat_df["ProductCategoryKey"].isin( productcat_df["ProductCategoryKey"]).all(), "Eén of meerdere ProductCategoryKeys uit ProductSubCategory bestaan niet in ProductCategory."
# Controleert of iedere ProductCategoryKey uit de subcategorie dataset voorkomt in de ProductCategory dataset.

# 8 ***Analyse Channel data***

## 8.1 Data importeren

In [ ]:
channel_path= Path(r"Data\Contoso_Channel.csv")
#Inlezen product channel.csv
channel_df = pd.read_csv(channel_path)
#Decimal weggehaald, gezien set met . werkt voor decimalen.

## 8.2 Data analyse

In [ ]:
init_analyse(channel_df)

De Channel dataset lijkt weinig gegevens te bevatten, enkel over de aanwezige kanalen.
In alle andere datasets wordt de key specifiek benoemd, hier staat enkel "Channel".
Dit dient nog gewijzigd te worden om consistent te blijven met de andere datasets.

## 8.3 Data Cleaning

In [ ]:
channel_df = channel_df.rename(columns={"Channel":"ChannelKey"})
#Channel hernoemd naar ChannelKey.

## 8.4 Validatie

In [ ]:
assert channel_df["ChannelKey"].is_unique, "ChannelKey bevat één of meerdere duplicaten."
# Controleert of alle ChannelKeys uniek zijn.

assert channel_df["ChannelKey"].notna().all(),  "ChannelKey bevat één of meerdere ontbrekende waarden."
# Controleert of iedere rij een ChannelKey bevat.

Dit onderdeel is bedoeld om onderlinge regels tussen de verschillende datasets te evalueren, alvorens deze op te slaan als parquet-bestanden die in PowerBI geladen zullen worden.

# 9 ***GDP per capita/ BBP per hoofd per jaar toevoegen(Worldbank dataset)***

Van de worldbank is een dataset gedownload met betrekking tot gegevens van jaar tot jaar over GDP per capita, oftewel bruto binnenlands product per hoofd van de bevolking.  
Het meet indirect hoe welvarend een land is.
Met deze gegevens kan er vergeleken worden hoe sterk omzet afhankelijk is van de welvaart van een land. 


## 9.1 Importen van data

In [ ]:
GDP_path= Path(r"Data\Contoso_GDP.csv")
#Inlezen product channel.csv
GDP_df = pd.read_csv(GDP_path,
                     skiprows=4)
#Eerste 4 rijen betreffen geen relevante data

## 9.2 Initiële analyse

In [ ]:
init_analyse(GDP_df)

Er zitten een hoop kolommen in de set die we niet nodig hebben in verband met de historische data over voorgaande jaren.
Deze dienen we eerst te beperken voordat we verder gaan. 

In [ ]:
GDP_df = GDP_df[["Country Name","Country Code","Indicator Name","Indicator Code","2011","2012","2013"]]

In [ ]:

GDP_df = GDP_df.melt(
   id_vars=["Country Name", "Country Code"],
   value_vars=["2011", "2012", "2013"],
   var_name="Year",
    value_name="GDPPerCapita"
)


In [ ]:
GDP_df

In [ ]:
GDP_df["Year"] = GDP_df["Year"].astype("int64")

In [ ]:
geography_df2["RegionCountryName"].isin(GDP_df["Country Name"]).value_counts()

In [ ]:
display(geography_df2[~geography_df2["RegionCountryName"].isin(GDP_df["Country Name"])]["RegionCountryName"].unique())

Lijst van landen die niet direct in de dataset gevonden worden.

In [ ]:
test = list(geography_df2[~(geography_df2["RegionCountryName"].isin(GDP_df["Country Name"]))]["RegionCountryName"].unique())
#Lijst maken met ontbrekende landen

In [ ]:
test = "|".join(test)
#Lijst joinen tot een string met een pipe tussen de woorden voor een or statement in str.contains.


In [ ]:
GDP_df[GDP_df["Country Name"].str.contains(test)]["Country Name"]
#Kijken of er landen in de dataset zitten die delen van het woord bevatten.

Drie landen gevonden die anders benoemd worden:
- Iran -> Iran, Islamic Rep.
- Russia -> Russian Federation
- Syria -> Syrian Arab Republic

Resteert 
- Kyrgyzstan
- South Korea
- Taiwan
- The Netherlands

In [ ]:
restant="Kyrgy|Korea|Taiwan|Nether"
#Lijst met waarden die nog ontbreken.
display(GDP_df[GDP_df["Country Name"].str.contains(restant)]["Country Name"])
#Controleren op resterende waardes.

2 waarden die Korea bevatten, Korea, Dem. People's Rep. betreft Noord Korea en bevat ook geen data.  
Eerder is de originele dataframe al overschreven, daarom doen we eerst een nieuwe import.  
Waarden gevonden voor alles behalve Taiwan. Taiwan staat ook bekend als Republic of China.  
Hier nog een extra check op doen. 

In [ ]:
display(GDP_df[GDP_df["Country Name"].str.contains("China")]["Country Name"])
#Kijken of republic of china wel aanwezig is.

Alleen voor Taiwan hebben we geen gegevens. De andere landen kunnen we koppelen, gezien de omvang van de dataset is dit acceptabel.


##  9.3 Data cleaning

In [ ]:
mapping = {"Kyrgyz Republic":"Kyrgyzstan",
           "Iran, Islamic Rep.":"Iran",
           "Russian Federation":"Russia",
           "Syrian Arab Republic":"Syria",
           "Korea, Rep.":"South Korea",
           "Netherlands":"the Netherlands",
           "Taiwan":pd.NA
           }

In [ ]:
GDP_df["CountryKey"] = GDP_df["Country Name"].replace(mapping)
#Vervang de namen in de huidige GDP Dataframe met de gekoppelde namen in de dictionary.

In [ ]:
geography_df2[~geography_df2["RegionCountryName"].isin(GDP_df["CountryKey"])]
#Controleren welke waarden niet voorkomen in GDP. Enkel nog Taiwan zoals verwacht

De originele namen zijn gemapt aan de waarden uit de geography dataset om deze als key te gebruiken. 
Vervolgens is er een nieuwe CountryKey kolom gemaakt om te linken op RegionCountryName.

## 9.4 Validatie

In [ ]:
assert geography_df["GeographyKey"].is_unique

# 10 ***Integrale validatie***

In [ ]:
assert sales_df2["ProductKey"].isin(product_df["ProductKey"]).all(), "Niet alle ProductKeys uit Sales bestaan in Product."
# Controleert de relatie Sales -> Product.

assert sales_df2["DateKey"].isin(calendar_df2["DateKey"]).all(), "Niet alle DateKeys uit Sales bestaan in Calendar."
# Controleert de relatie Sales -> Calendar.

assert sales_df2["StoreKey"].isin(stores_df["StoreKey"]).all(), "Niet alle StoreKeys uit Sales bestaan in Stores."
# Controleert de relatie Sales -> Stores.

assert sales_df2["PromotionKey"].isin(promotions_df["PromotionKey"]).all(), "Niet alle PromotionKeys uit Sales bestaan in Promotions."
# Controleert de relatie Sales -> Promotions.

assert sales_df2["ChannelKey"].isin(channel_df["ChannelKey"]).all(), "Niet alle ChannelKeys uit Sales bestaan in Channel."
# Controleert de relatie Sales -> Channel.

assert stores_df["GeographyKey"].isin(geography_df2["GeographyKey"]).all(), "Niet alle GeographyKeys uit Stores bestaan in Geography."
# Controleert de relatie Stores -> Geography.

assert product_df["ProductSubcategoryKey"].isin(productsubcat_df["ProductSubcategoryKey"]).all(), "Niet alle ProductSubcategoryKeys uit Product bestaan in ProductSubCategory."
# Controleert de relatie Product -> ProductSubCategory.


# 11 ***Opslaan***

In [ ]:
sales_df2.to_parquet(r"PQ/Contoso_sales.parquet")
product_df.to_parquet(r"PQ/Contoso_product.parquet")
calendar_df2.to_parquet(r"PQ/Contoso_calendar.parquet")
geography_df2.to_parquet(r"PQ/Contoso_geography.parquet")
stores_df.to_parquet(r"PQ/Contoso_stores.parquet")
promotions_df.to_parquet(r"PQ/Contoso_promotions.parquet")
productmerge_df.to_parquet(r"PQ/Contoso_productcat.parquet")
channel_df.to_parquet(r"PQ/Contoso_channel.parquet")
GDP_df.to_parquet(r"PQ/Contose_GDP.parquet")